In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path(
    "data/ohsumed-all"
)

documents = {}

for category_dir in sorted(DATA_DIR.glob("C*")):
    if not category_dir.is_dir():
        continue

    label = category_dir.name

    for file_path in category_dir.iterdir():
        if not file_path.is_file():
            continue

        document_id = file_path.name

        text = file_path.read_text(
            encoding="latin-1",
            errors="replace",
        ).strip()

        if document_id not in documents:
            documents[document_id] = {
                "document_id": document_id,
                "text": text,
                "labels": set(),
            }

        documents[document_id]["labels"].add(label)


df = pd.DataFrame(documents.values())

# Convert each label set into a sorted list
df["labels"] = df["labels"].apply(
    lambda x: sorted(x)
)

df = (
    df.sort_values("document_id")
    .reset_index(drop=True)
)

print(df.head())
print("Unique documents:", len(df))
print(
    "Unique labels:",
    df["labels"].explode().nunique(),
)
print(
    "Total document-label assignments:",
    df["labels"].str.len().sum(),
)
print(
    "Average labels per document:",
    df["labels"].str.len().mean(),
)

  document_id                                               text      labels
0     0000000  Evaluation of women with possible appendicitis...       [C06]
1     0000001  Cause of death in an emergency department \n A...       [C14]
2     0000002  Treatment of Fournier's gangrene with adjuncti...  [C12, C23]
3     0000003  Intermittent obstruction of an incarcerated hi...       [C06]
4     0000005  Excitatory amino acids in the developing brain...       [C10]
Unique documents: 34389
Unique labels: 23
Total document-label assignments: 56984
Average labels per document: 1.657041495827154


In [2]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Unique document IDs:", df["document_id"].nunique())
print("Unique labels:", df["labels"].explode().nunique())
print("Total label assignments:", df["labels"].str.len().sum())

Shape: (34389, 3)
Columns: ['document_id', 'text', 'labels']
Unique document IDs: 34389
Unique labels: 23
Total label assignments: 56984


In [3]:
df.head()

,document_id,text,labels
0,0000000,Evaluation of women with possible appendicitis...,[C06]
1,0000001,Cause of death in an emergency department \n A...,[C14]
2,0000002,Treatment of Fournier's gangrene with adjuncti...,"[C12, C23]"
3,0000003,Intermittent obstruction of an incarcerated hi...,[C06]
4,0000005,Excitatory amino acids in the developing brain...,[C10]


In [4]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

Shape: (34389, 3)

Missing values:
document_id    0
text           0
labels         0
dtype: int64


In [5]:
print(
    "Missing document IDs:",
    df["document_id"].isna().sum(),
)

print(
    "Missing texts:",
    df["text"].isna().sum(),
)

print(
    "Missing labels:",
    df["labels"].isna().sum(),
)

Missing document IDs: 0
Missing texts: 0
Missing labels: 0


In [6]:
empty_document_ids = (
    df["document_id"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

empty_texts = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

print(
    "Empty document IDs:",
    empty_document_ids.sum(),
)

print(
    "Empty texts:",
    empty_texts.sum(),
)

Empty document IDs: 0
Empty texts: 0


In [7]:
df["_text_key"] = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

In [8]:
duplicate_text_mask = (
    df["_text_key"]
    .duplicated(keep=False)
)

print(
    "Rows with duplicated text:",
    duplicate_text_mask.sum(),
)

print(
    "Number of duplicated text groups:",
    df.loc[
        duplicate_text_mask,
        "_text_key",
    ].nunique(),
)

Rows with duplicated text: 6
Number of duplicated text groups: 3


In [9]:
with pd.option_context(
    "display.max_colwidth",
    200,
):
    display(
        df.loc[
            duplicate_text_mask,
            [
                "document_id",
                "labels",
                "_text_key",
            ],
        ]
        .sort_values("_text_key")
        .head(20)
    )

,document_id,labels,_text_key
33549,0048731,"[C10, C23]","Detection of preclinical Parkinson's disease with PET. Putamen 18F-dopa uptake of patients with Parkinson's disease (PD) is reduced by at least 35% at onset of symptoms; therefore, positron-emissi..."
34332,0050118,"[C10, C23]","Detection of preclinical Parkinson's disease with PET. Putamen 18F-dopa uptake of patients with Parkinson's disease (PD) is reduced by at least 35% at onset of symptoms; therefore, positron-emissi..."
30810,0044187,"[C10, C23]",Electrophysiologic analysis of early Parkinson's disease. We have been interested in the application of quantitative measures of motor performance as a possible means of early detection of Parkins...
31356,0044981,"[C10, C23]",Electrophysiologic analysis of early Parkinson's disease. We have been interested in the application of quantitative measures of motor performance as a possible means of early detection of Parkins...
21523,0030921,"[C10, C23]","Genetic susceptibility to Parkinson's disease. Genetic factors clearly cause Lewy-body Parkinson's disease (PD) in a subset of autosomal-dominant families. However, most cases of PD are sporadic. ..."
30814,0044191,"[C10, C23]","Genetic susceptibility to Parkinson's disease. Genetic factors clearly cause Lewy-body Parkinson's disease (PD) in a subset of autosomal-dominant families. However, most cases of PD are sporadic. ..."


In [10]:
from itertools import chain


def merge_label_lists(label_series):
    return sorted(
        set(
            chain.from_iterable(
                label_series
            )
        )
    )


df = (
    df.groupby(
        "_text_key",
        as_index=False,
    )
    .agg(
        document_id=(
            "document_id",
            "first",
        ),
        text=(
            "text",
            "first",
        ),
        labels=(
            "labels",
            merge_label_lists,
        ),
    )
)

In [11]:
label_type_counts = (
    df["labels"]
    .apply(type)
    .value_counts()
)

print(label_type_counts)

labels
<class 'list'>    34386
Name: count, dtype: int64


In [12]:
empty_labels = df["labels"].apply(
    lambda x: (
        isinstance(x, list)
        and len(x) == 0
    )
)

print(
    "Documents without labels:",
    empty_labels.sum(),
)

Documents without labels: 0


In [13]:
df = df.copy()

# Remove rows with missing essential values
df = df.dropna(
    subset=[
        "document_id",
        "text",
        "labels",
    ]
)

# Minimal text normalization
df["training_text"] = (
    df["text"]
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

# Remove empty texts
df = df[
    df["training_text"].ne("")
]

# Ensure sorted and unique labels
df["labels"] = df["labels"].apply(
    lambda labels: sorted(
        set(labels)
    )
)

# Remove empty label rows
df = df[
    df["labels"].str.len() > 0
]

# Remove duplicate document IDs
df = df.drop_duplicates(
    subset=["document_id"],
    keep="first",
)

df = df.reset_index(drop=True)

print("Final shape:", df.shape)

Final shape: (34386, 5)


In [14]:
expected_labels = {
    f"C{i:02d}"
    for i in range(1, 24)
}

In [15]:
assert df["document_id"].notna().all()
assert df["document_id"].is_unique

assert df["training_text"].notna().all()
assert df["training_text"].str.len().gt(0).all()

assert df["labels"].notna().all()
assert df["labels"].map(
    lambda x: isinstance(x, list)
).all()
assert df["labels"].str.len().gt(0).all()

assert set(
    df["labels"].explode().unique()
) == expected_labels

print("Dataset validation passed.")

Dataset validation passed.


In [16]:
expected_labels = {
    f"C{i:02d}"
    for i in range(1, 24)
}

observed_labels = set(
    df["labels"]
    .explode()
    .dropna()
    .unique()
)

assert df["document_id"].notna().all()
assert df["document_id"].is_unique

assert df["training_text"].notna().all()
assert df["training_text"].str.len().gt(0).all()

assert df["labels"].notna().all()

assert df["labels"].map(
    lambda x: isinstance(x, list)
).all()

assert df["labels"].str.len().gt(0).all()

assert observed_labels == expected_labels

print("Dataset validation passed.")
print("Documents:", len(df))
print("Observed labels:", sorted(observed_labels))

Dataset validation passed.
Documents: 34386
Observed labels: ['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23']


In [17]:
training_df = df[
    [
        "document_id",
        "training_text",
        "labels",
    ]
].copy()

print(training_df.shape)
display(training_df.head())

(34386, 3)


,document_id,training_text,labels
0,0042801,"""Angioglioma"" and the arteriovenous malformati...",[C04]
1,0042146,"""Beaten to death"". An autopsy series of homici...",[C21]
2,0006344,"""Cloggology"" revisited: endoscopic or surgical...",[C23]
3,0031640,"""Designer drugs""--a current perspective. Since...",[C21]
4,0041071,"""Directed"" emergency room thoracotomy: a progn...",[C21]


In [18]:
training_df.to_parquet(
    "data/ohsumed_training_dataframe.parquet",
    index=False,
)

In [19]:
full_df = df[
    [
        "document_id",
        "training_text",
        "labels",
    ]
].copy()

In [20]:
import numpy as np

from sklearn.preprocessing import (
    MultiLabelBinarizer
)

label_names = [
    f"C{i:02d}"
    for i in range(1, 24)
]

mlb = MultiLabelBinarizer(
    classes=label_names
)

label_matrix = mlb.fit_transform(
    full_df["labels"]
).astype(np.float32)

print("Documents:", len(full_df))
print(
    "Label matrix:",
    label_matrix.shape,
)
print("Label order:", mlb.classes_)

Documents: 34386
Label matrix: (34386, 23)
Label order: ['C01' 'C02' 'C03' 'C04' 'C05' 'C06' 'C07' 'C08' 'C09' 'C10' 'C11' 'C12'
 'C13' 'C14' 'C15' 'C16' 'C17' 'C18' 'C19' 'C20' 'C21' 'C22' 'C23']


In [21]:
%pip install iterative-stratification

Note: you may need to restart the kernel to use updated packages.


In [22]:
from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

all_indices = np.arange(
    len(full_df)
)

# 80% train and 20% temporary
first_splitter = (
    MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=42,
    )
)

train_indices, temporary_indices = next(
    first_splitter.split(
        all_indices,
        label_matrix,
    )
)

# Divide temporary data equally:
# 10% validation and 10% test
second_splitter = (
    MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=0.50,
        random_state=42,
    )
)

validation_relative, test_relative = next(
    second_splitter.split(
        temporary_indices,
        label_matrix[
            temporary_indices
        ],
    )
)

validation_indices = temporary_indices[
    validation_relative
]

test_indices = temporary_indices[
    test_relative
]

In [23]:
train_df = (
    full_df
    .iloc[train_indices]
    .reset_index(drop=True)
)

validation_df = (
    full_df
    .iloc[validation_indices]
    .reset_index(drop=True)
)

test_df = (
    full_df
    .iloc[test_indices]
    .reset_index(drop=True)
)

print("Full:", len(full_df))
print("Train:", len(train_df))
print(
    "Validation:",
    len(validation_df),
)
print("Test:", len(test_df))

Full: 34386
Train: 27554
Validation: 3449
Test: 3383


In [24]:
train_df = (
    full_df
    .iloc[train_indices]
    .reset_index(drop=True)
)

validation_df = (
    full_df
    .iloc[validation_indices]
    .reset_index(drop=True)
)

test_df = (
    full_df
    .iloc[test_indices]
    .reset_index(drop=True)
)

train_df.to_parquet(
    "data/ohsumed_train.parquet",
    index=False,
)

validation_df.to_parquet(
    "data/ohsumed_validation.parquet",
    index=False,
)

test_df.to_parquet(
    "data/ohsumed_test.parquet",
    index=False,
)

In [25]:
train_label_matrix = (
    label_matrix[train_indices]
    .astype(np.float32)
)

validation_label_matrix = (
    label_matrix[
        validation_indices
    ]
    .astype(np.float32)
)

test_label_matrix = (
    label_matrix[test_indices]
    .astype(np.float32)
)

print(
    "Train label matrix:",
    train_label_matrix.shape,
)

print(
    "Validation label matrix:",
    validation_label_matrix.shape,
)

print(
    "Test label matrix:",
    test_label_matrix.shape,
)

Train label matrix: (27554, 23)
Validation label matrix: (3449, 23)
Test label matrix: (3383, 23)


In [26]:
from datasets import Dataset, DatasetDict


def make_hf_dataset(
    dataframe,
    labels,
):
    return Dataset.from_dict({
        "text": (
            dataframe["training_text"]
            .tolist()
        ),
        "labels": (
            labels
            .astype(np.float32)
            .tolist()
        ),
    })


raw_dataset = DatasetDict({
    "train": make_hf_dataset(
        train_df,
        train_label_matrix,
    ),
    "validation": make_hf_dataset(
        validation_df,
        validation_label_matrix,
    ),
    "test": make_hf_dataset(
        test_df,
        test_label_matrix,
    ),
})

print(raw_dataset)
print(raw_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 27554
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 3449
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 3383
    })
})
{'text': '"Angioglioma" and the arteriovenous malformation-glioma association. The term "angioglioma" denotes a highly vascular glioma, most of which are low-grade lesions associated with a favorable prognosis. The authors encountered an example of this pathology, a cystic oligodendroglioma associated with prominent vasculature which both clinically and histologically mimicked an occult arteriovenous malformation (AVM). This case and reports of the association of AVM and glioma prompted a histological review of 1034 surgically resected AVM\'s, both angiographically occult and visible, among which no oligodendroglial or astrocytic forms of "angioglioma" were found. Eight cases were observed, however, wh

In [27]:
import os
import gc
import time
import numpy as np
import pandas as pd
import torch

from scipy.special import expit

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)


MODELS = [
    "google-bert/bert-base-uncased",
    "allenai/scibert_scivocab_uncased",
    "nlpaueb/legal-bert-base-uncased",
    "ahmedrachid/FinancialBERT",
    "Lianglab/PharmBERT-uncased",
    "recobo/agriculture-bert-uncased",
    "recobo/chemical-bert-uncased",
]

NUM_LABELS = 23
N_SPLITS = 5
TRAINING_SEED = 42

NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# 必须与之前EnvDataset一致
MAX_LENGTH = 128

OUTPUT_ROOT = (
    "results/ohsumed_5fold"
)

FOLD_RESULT_CSV = os.path.join(
    OUTPUT_ROOT,
    "ohsumed_5fold_results.csv",
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True,
)

In [ ]:
def calculate_multilabel_metrics(
    logits,
    true_labels,
    threshold=0.5,
):
    probabilities = expit(logits)

    predicted_labels = (
        probabilities >= threshold
    ).astype(int)

    true_labels = (
        true_labels.astype(int)
    )

    return {
        # Multi-label accuracy_score
        # means exact/subset accuracy
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels,
        ),

        "macro_f1": f1_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),

        "micro_f1": f1_score(
            true_labels,
            predicted_labels,
            average="micro",
            zero_division=0,
        ),

        "macro_precision":
            precision_score(
                true_labels,
                predicted_labels,
                average="macro",
                zero_division=0,
            ),

        "macro_recall":
            recall_score(
                true_labels,
                predicted_labels,
                average="macro",
                zero_division=0,
            ),
    }

In [29]:
def evaluate_model_5fold_multilabel(
    model_name,
    full_df,
    label_matrix,
    n_splits=5,
):
    print("\nEvaluating:", model_name)

    splitter = (
        MultilabelStratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=42,
        )
    )

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            model_name,
            use_fast=True,
        )
    )

    # Tokenize the complete dataset once
    complete_dataset = Dataset.from_dict({
        "text": (
            full_df["training_text"]
            .tolist()
        ),
        "labels": (
            label_matrix
            .astype(np.float32)
            .tolist()
        ),
    })

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    tokenized_dataset = (
        complete_dataset.map(
            tokenize_batch,
            batched=True,
            remove_columns=["text"],
            desc=f"Tokenizing {model_name}",
        )
    )

    tokenized_dataset.set_format(
        type="torch"
    )

    data_collator = (
        DataCollatorWithPadding(
            tokenizer=tokenizer,
            padding="longest",
        )
    )

    fold_results = []

    for fold, (
        train_indices,
        test_indices,
    ) in enumerate(
        splitter.split(
            np.zeros(
                (len(full_df), 1)
            ),
            label_matrix,
        ),
        start=1,
    ):
        print(
            f"\n========== "
            f"Fold {fold}/{n_splits} "
            f"=========="
        )

        set_seed(TRAINING_SEED)

        train_dataset = (
            tokenized_dataset.select(
                train_indices.tolist()
            )
        )

        test_dataset = (
            tokenized_dataset.select(
                test_indices.tolist()
            )
        )

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                model_name,
                num_labels=NUM_LABELS,
                problem_type=(
                    "multi_label_classification"
                ),
            )
        )

        safe_model_name = (
            model_name
            .replace("/", "__")
        )

        output_dir = os.path.join(
            OUTPUT_ROOT,
            (
                f"{safe_model_name}"
                f"_fold_{fold}"
            ),
        )

        training_args = (
            TrainingArguments(
                output_dir=output_dir,

                eval_strategy="no",
                save_strategy="no",

                logging_steps=50,

                per_device_train_batch_size=(
                    TRAIN_BATCH_SIZE
                ),

                per_device_eval_batch_size=(
                    EVAL_BATCH_SIZE
                ),

                num_train_epochs=(
                    NUM_EPOCHS
                ),

                learning_rate=(
                    LEARNING_RATE
                ),

                weight_decay=(
                    WEIGHT_DECAY
                ),

                seed=TRAINING_SEED,
                data_seed=TRAINING_SEED,

                report_to="none",

                # Prevent notebook callback error
                disable_tqdm=True,

                fp16=False,
                bf16=False,

                use_cpu=(
                    not torch.cuda.is_available()
                ),
            )
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
        )

        start_time = time.time()

        train_result = trainer.train()

        predictions = trainer.predict(
            test_dataset
        )

        logits = predictions.predictions

        if isinstance(logits, tuple):
            logits = logits[0]

        metrics = (
            calculate_multilabel_metrics(
                logits=logits,
                true_labels=(
                    predictions.label_ids
                ),
                threshold=0.5,
            )
        )

        elapsed_hours = (
            time.time() - start_time
        ) / 3600

        fold_row = {
            "dataset": "ohsumed",
            "model": model_name,
            "fold": fold,

            "accuracy":
                metrics["accuracy"],

            "macro_f1":
                metrics["macro_f1"],

            "micro_f1":
                metrics["micro_f1"],

            "macro_precision":
                metrics[
                    "macro_precision"
                ],

            "macro_recall":
                metrics[
                    "macro_recall"
                ],

            "training_loss":
                train_result.training_loss,

            "training_hours":
                elapsed_hours,

            "training_seed":
                TRAINING_SEED,

            "epochs":
                NUM_EPOCHS,

            "max_length":
                MAX_LENGTH,

            "train_batch_size":
                TRAIN_BATCH_SIZE,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "threshold":
                0.5,

            "train_size":
                len(train_dataset),

            "test_size":
                len(test_dataset),
        }

        fold_results.append(
            fold_row
        )

        print(
            f"Fold {fold} Accuracy: "
            f"{metrics['accuracy']:.4f}"
        )

        print(
            f"Fold {fold} Macro F1: "
            f"{metrics['macro_f1']:.4f}"
        )

        print(
            f"Fold {fold} Micro F1: "
            f"{metrics['micro_f1']:.4f}"
        )

        print(
            f"Fold {fold} Hours: "
            f"{elapsed_hours:.2f}"
        )

        # Save immediately after every fold
        new_fold_df = pd.DataFrame(
            fold_results
        )

        del model
        del trainer
        del train_dataset
        del test_dataset

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_df = pd.DataFrame(
        fold_results
    )

    return fold_df

In [30]:
if os.path.exists(
    FOLD_RESULT_CSV
):
    all_fold_results = pd.read_csv(
        FOLD_RESULT_CSV
    )
else:
    all_fold_results = (
        pd.DataFrame()
    )


for model_name in MODELS:
    completed_folds = set()

    if not all_fold_results.empty:
        model_previous_results = (
            all_fold_results[
                all_fold_results["model"]
                == model_name
            ]
        )

        completed_folds = set(
            model_previous_results[
                "fold"
            ].astype(int)
        )

    if len(completed_folds) == N_SPLITS:
        print(
            "Skipping completed model:",
            model_name,
        )
        continue

    model_fold_df = (
        evaluate_model_5fold_multilabel(
            model_name=model_name,
            full_df=full_df,
            label_matrix=label_matrix,
            n_splits=N_SPLITS,
        )
    )

    # Remove older rows for this model
    if not all_fold_results.empty:
        all_fold_results = (
            all_fold_results[
                all_fold_results["model"]
                != model_name
            ]
        )

    all_fold_results = pd.concat(
        [
            all_fold_results,
            model_fold_df,
        ],
        ignore_index=True,
    )

    all_fold_results.to_csv(
        FOLD_RESULT_CSV,
        index=False,
    )

    print(
        "Saved:",
        FOLD_RESULT_CSV,
    )

Skipping completed model: google-bert/bert-base-uncased
Skipping completed model: allenai/scibert_scivocab_uncased
Skipping completed model: nlpaueb/legal-bert-base-uncased
Skipping completed model: ahmedrachid/FinancialBERT
Skipping completed model: Lianglab/PharmBERT-uncased
Skipping completed model: recobo/agriculture-bert-uncased
Skipping completed model: recobo/chemical-bert-uncased


In [31]:
if os.path.exists(
    FOLD_RESULT_CSV
):
    all_fold_results = pd.read_csv(
        FOLD_RESULT_CSV
    )
else:
    all_fold_results = (
        pd.DataFrame()
    )


for model_name in MODELS:
    completed_folds = set()

    if not all_fold_results.empty:
        model_previous_results = (
            all_fold_results[
                all_fold_results["model"]
                == model_name
            ]
        )

        completed_folds = set(
            model_previous_results[
                "fold"
            ].astype(int)
        )

    if len(completed_folds) == N_SPLITS:
        print(
            "Skipping completed model:",
            model_name,
        )
        continue

    model_fold_df = (
        evaluate_model_5fold_multilabel(
            model_name=model_name,
            full_df=full_df,
            label_matrix=label_matrix,
            n_splits=N_SPLITS,
        )
    )

    # Remove older rows for this model
    if not all_fold_results.empty:
        all_fold_results = (
            all_fold_results[
                all_fold_results["model"]
                != model_name
            ]
        )

    all_fold_results = pd.concat(
        [
            all_fold_results,
            model_fold_df,
        ],
        ignore_index=True,
    )

    all_fold_results.to_csv(
        FOLD_RESULT_CSV,
        index=False,
    )

    print(
        "Saved:",
        FOLD_RESULT_CSV,
    )

Skipping completed model: google-bert/bert-base-uncased
Skipping completed model: allenai/scibert_scivocab_uncased
Skipping completed model: nlpaueb/legal-bert-base-uncased
Skipping completed model: ahmedrachid/FinancialBERT
Skipping completed model: Lianglab/PharmBERT-uncased
Skipping completed model: recobo/agriculture-bert-uncased
Skipping completed model: recobo/chemical-bert-uncased


In [32]:
import pandas as pd

# Read saved 5-fold results
all_fold_results = pd.read_csv(
    FOLD_RESULT_CSV
)

# Average the five folds for each model
real_model_ranking = (
    all_fold_results
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        completed_folds=(
            "fold",
            "nunique",
        ),

        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),
        macro_f1_std=(
            "macro_f1",
            "std",
        ),

        micro_f1_mean=(
            "micro_f1",
            "mean",
        ),
        micro_f1_std=(
            "micro_f1",
            "std",
        ),

        accuracy_mean=(
            "accuracy",
            "mean",
        ),
        accuracy_std=(
            "accuracy",
            "std",
        ),

        macro_precision_mean=(
            "macro_precision",
            "mean",
        ),

        macro_recall_mean=(
            "macro_recall",
            "mean",
        ),

        training_hours_total=(
            "training_hours",
            "sum",
        ),
    )
)

# Keep only models with all five folds
real_model_ranking = (
    real_model_ranking[
        real_model_ranking[
            "completed_folds"
        ] == N_SPLITS
    ]
    .copy()
)

# Rank by mean Macro-F1
real_model_ranking[
    "real_rank"
] = (
    real_model_ranking[
        "macro_f1_mean"
    ]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

# Sort by real rank
real_model_ranking = (
    real_model_ranking
    .sort_values(
        "real_rank"
    )
    .reset_index(drop=True)
)

# Put rank first
real_model_ranking = (
    real_model_ranking[
        [
            "real_rank",
            "model",
            "macro_f1_mean",
            "macro_f1_std",
            "micro_f1_mean",
            "micro_f1_std",
            "accuracy_mean",
            "accuracy_std",
            "macro_precision_mean",
            "macro_recall_mean",
            "completed_folds",
            "training_hours_total",
        ]
    ]
)

display(real_model_ranking)

,real_rank,model,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,accuracy_mean,accuracy_std,macro_precision_mean,macro_recall_mean,completed_folds,training_hours_total
0,1,recobo/agriculture-bert-uncased,0.729418,0.003852,0.768456,0.002379,0.519483,0.003197,0.794283,0.679881,5,15.636197
1,2,allenai/scibert_scivocab_uncased,0.727792,0.005204,0.766890,0.002803,0.518516,0.002442,0.795794,0.675790,5,16.169732
2,3,Lianglab/PharmBERT-uncased,0.682587,0.004248,0.737058,0.002525,0.486205,0.003862,0.792773,0.612402,5,15.593336
3,4,ahmedrachid/FinancialBERT,0.608793,0.006484,0.694457,0.002739,0.434559,0.003821,0.735171,0.535554,5,15.470806
4,5,google-bert/bert-base-uncased,0.597272,0.006811,0.709638,0.001588,0.455286,0.003623,0.781466,0.522738,5,15.858653
5,6,recobo/chemical-bert-uncased,0.582626,0.005372,0.678927,0.002444,0.415572,0.003651,0.716180,0.510144,5,15.544119
6,7,nlpaueb/legal-bert-base-uncased,0.577278,0.003937,0.689693,0.001361,0.433097,0.004499,0.771329,0.501589,5,15.543872


In [33]:
REAL_RANKING_CSV = os.path.join(
    OUTPUT_ROOT,
    "ohsumed_real_model_ranking.csv",
)

real_model_ranking.to_csv(
    REAL_RANKING_CSV,
    index=False,
)

print("Saved to:", REAL_RANKING_CSV)

Saved to: results/ohsumed_5fold/ohsumed_real_model_ranking.csv


In [34]:
display(
    all_fold_results
    .groupby("model")["fold"]
    .agg(
        [
            "count",
            "nunique",
        ]
    )
)

,count,nunique
model,,
Lianglab/PharmBERT-uncased,5,5
ahmedrachid/FinancialBERT,5,5
allenai/scibert_scivocab_uncased,5,5
google-bert/bert-base-uncased,5,5
nlpaueb/legal-bert-base-uncased,5,5
recobo/agriculture-bert-uncased,5,5
recobo/chemical-bert-uncased,5,5


In [35]:
selected_ranks = [
    1,
    2,
    4,
    6,
    7,
]

selected_ranking = (
    real_model_ranking[
        real_model_ranking[
            "real_rank"
        ].isin(selected_ranks)
    ]
    .sort_values("real_rank")
    .reset_index(drop=True)
)

selected_models = (
    selected_ranking["model"]
    .tolist()
)

model_real_ranks = dict(
    zip(
        selected_ranking["model"],
        selected_ranking["real_rank"],
    )
)

display(
    selected_ranking[
        [
            "real_rank",
            "model",
            "macro_f1_mean",
            "macro_f1_std",
        ]
    ]
)

,real_rank,model,macro_f1_mean,macro_f1_std
0,1,recobo/agriculture-bert-uncased,0.729418,0.003852
1,2,allenai/scibert_scivocab_uncased,0.727792,0.005204
2,4,ahmedrachid/FinancialBERT,0.608793,0.006484
3,6,recobo/chemical-bert-uncased,0.582626,0.005372
4,7,nlpaueb/legal-bert-base-uncased,0.577278,0.003937


In [36]:
short_model_names = {
    "recobo/agriculture-bert-uncased":
        "AgricultureBERT",

    "allenai/scibert_scivocab_uncased":
        "SciBERT",

    "nlpaueb/legal-bert-base-uncased":
        "LegalBERT",

    "ahmedrachid/FinancialBERT":
        "FinancialBERT",

    "Lianglab/PharmBERT-uncased":
        "PharmBERT",

    "recobo/chemical-bert-uncased":
        "ChemicalBERT",

    "google-bert/bert-base-uncased":
        "BERT",
}

In [37]:
minimum_rank = 1
maximum_rank = 7

model_colours = {}

for model_name in selected_models:
    real_rank = model_real_ranks[
        model_name
    ]

    rank_position = (
        real_rank - minimum_rank
    ) / (
        maximum_rank - minimum_rank
    )

    # Rank 1 -> 0.90: darkest
    # Rank 7 -> 0.45: lightest
    red_intensity = (
        0.90
        - 0.45 * rank_position
    )

    model_colours[model_name] = (
        plt.cm.Reds(red_intensity)
    )

In [38]:
import matplotlib.pyplot as plt
def plot_ranked_distributions(
    selected_models,
    distribution_data,
    score_key,
    mean_key,
    eta_key,
    title,
    output_file,
):
    plt.figure(
        figsize=(12, 7)
    )

    for model_name in selected_models:
        values = distribution_data[
            model_name
        ]

        scores = values[score_key]
        mean_value = values[mean_key]
        eta_value = values[eta_key]

        real_rank = model_real_ranks[
            model_name
        ]

        colour = model_colours[
            model_name
        ]

        model_label = (
            f"Rank {real_rank}: "
            f"{short_model_names[model_name]} "
            f"(mean={mean_value:.4f}, "
            f"eta={eta_value:.4f})"
        )

        sns.kdeplot(
            scores,
            color=colour,
            linewidth=2.7,
            bw_adjust=0.8,
            clip=(-1, 1),
            cut=0,
            label=model_label,
        )

        # Mean position
        plt.axvline(
            mean_value,
            color=colour,
            linestyle="--",
            linewidth=1.8,
            alpha=0.9,
        )

    plt.xlim(
        display_minimum,
        display_maximum,
    )

    plt.xlabel(
        "Cosine Similarity"
    )

    plt.ylabel(
        "Density"
    )

    plt.title(
        title,
        fontsize=15,
        fontweight="bold",
    )

    plt.legend(
        fontsize=9,
        title=(
            "Darker red = better "
            "fine-tuning rank"
        ),
        title_fontsize=10,
    )

    plt.grid(
        alpha=0.2
    )

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

In [39]:
def plot_ranked_distributions(
    selected_models,
    distribution_data,
    score_key,
    mean_key,
    eta_key,
    title,
    output_file,
):
    plt.figure(
        figsize=(12, 7)
    )

    for model_name in selected_models:
        values = distribution_data[
            model_name
        ]

        scores = values[score_key]
        mean_value = values[mean_key]
        eta_value = values[eta_key]

        real_rank = model_real_ranks[
            model_name
        ]

        colour = model_colours[
            model_name
        ]

        model_label = (
            f"Rank {real_rank}: "
            f"{short_model_names[model_name]} "
            f"(mean={mean_value:.4f}, "
            f"eta={eta_value:.4f})"
        )

        sns.kdeplot(
            scores,
            color=colour,
            linewidth=2.7,
            bw_adjust=0.8,
            clip=(-1, 1),
            cut=0,
            label=model_label,
        )

        # Mean position
        plt.axvline(
            mean_value,
            color=colour,
            linestyle="--",
            linewidth=1.8,
            alpha=0.9,
        )

    plt.xlim(
        display_minimum,
        display_maximum,
    )

    plt.xlabel(
        "Cosine Similarity"
    )

    plt.ylabel(
        "Density"
    )

    plt.title(
        title,
        fontsize=15,
        fontweight="bold",
    )

    plt.legend(
        fontsize=9,
        title=(
            "Darker red = better "
            "fine-tuning rank"
        ),
        title_fontsize=10,
    )

    plt.grid(
        alpha=0.2
    )

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

In [43]:
distribution_data = {}

In [45]:
plot_ranked_distributions(
    selected_models=selected_models,
    distribution_data=distribution_data,
    score_key="same_scores",
    mean_key="same_mean",
    eta_key="same_eta",
    title=(
        "Same-label Cosine Similarity "
        "Distributions across Model Ranks"
    ),
    output_file=(
        "same_label_rank_spanning_models.png"
    ),
)

KeyError: 'recobo/agriculture-bert-uncased'

<Figure size 1200x700 with 0 Axes>

In [41]:
distribution_data = {}

for model_name in selected_models:
    model_runs = [
        run_result
        for (
            repetition,
            stored_model_name,
        ), run_result
        in results["full_results"].items()
        if stored_model_name == model_name
    ]

    if len(model_runs) == 0:
        raise ValueError(
            f"No distribution results found "
            f"for {model_name}"
        )

    combined_same_scores = np.concatenate([
        np.asarray(
            run_result["same_scores"]
        )
        for run_result in model_runs
    ])

    combined_diff_scores = np.concatenate([
        np.asarray(
            run_result[
                "diff_scores_original"
            ]
        )
        for run_result in model_runs
    ])

    distribution_data[model_name] = {
        "same_scores":
            combined_same_scores,

        "diff_scores":
            combined_diff_scores,

        "same_mean":
            combined_same_scores.mean(),

        "diff_mean":
            combined_diff_scores.mean(),

        "same_eta":
            asymmetry_eta(
                combined_same_scores
            ),

        "diff_eta":
            asymmetry_eta(
                combined_diff_scores
            ),
    }

NameError: name 'results' is not defined